# V22 E016 clean secondary checkpoint

This notebook trains the secondary detector without using the 27 development samples or their 46 labeled events. It is a training artifact only: no division assignment, lineage mutation, or full-cohort evaluation is enabled.

In [ ]:
from pathlib import Path
import json, hashlib, glob, os, shutil, subprocess, sys
import torch
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
ROOT = Path('/kaggle/working/Atabey')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', 'v22-upstream-division-availability', 'https://github.com/drosadocastro-bit/Atabey.git', str(ROOT)], check=True)
sys.path.insert(0, str(ROOT))
print('Repo:', ROOT)

In [ ]:
# Locate attached competition data and the public E016 training source.
train_candidates = [Path(p) for p in glob.glob('/kaggle/input/**/train', recursive=True)]
train_candidates = [p for p in train_candidates if len(list(p.glob('*.zarr'))) >= 190]
assert train_candidates, 'Could not find the 199-sample competition train directory'
TRAIN_DIR = train_candidates[0]
artifact_candidates = [Path(p) for p in glob.glob('/kaggle/input/**/repo/scripts/train_unet_transformer.py', recursive=True)]
assert artifact_candidates, 'Attach pilkwang/biohub-temporal-unet3d-seed314159-v1 as a notebook input'
PUBLIC_TRAINER = artifact_candidates[0]
print('Train:', TRAIN_DIR)
print('Samples:', len(list(TRAIN_DIR.glob('*.zarr'))))
print('Public trainer:', PUBLIC_TRAINER)

In [ ]:
# Install only the public trainer's runtime requirements if they are absent.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tracksdata', 'zarr>=3.0.10,<4', 'pyscipopt', 'geff', 'ilpy', 'polars', 'blosc2', 'dask', 'imagecodecs', 'pyarrow', 'rustworkx', 'sqlalchemy'], check=True)
print('Dependencies ready')

In [ ]:
# Build the immutable 138/34 split from the committed manifest.
split = json.loads((ROOT / 'v22_e016_clean_internal_split.json').read_text())
available = {p.name.removesuffix('.zarr'): p.name for p in TRAIN_DIR.glob('*.zarr')}
excluded = set(split['development_excluded_sample_ids'])
fit_ids = set(split['fit_sample_ids'])
val_ids = set(split['internal_validation_sample_ids'])
assert len(available) == 199 and len(excluded) == 27 and len(fit_ids) == 138 and len(val_ids) == 34
assert not (excluded & fit_ids) and not (excluded & val_ids) and not (fit_ids & val_ids)
assert {s.split('_', 1)[0] for s in val_ids} == {'44b6', '6bba'}
split_payload = [{'train': [available[s] for s in sorted(fit_ids)], 'test': [available[s] for s in sorted(val_ids)]}]
SPLITS = Path('/kaggle/working/dataset_splits_v22_e016_clean.json')
SPLITS.write_text(json.dumps(split_payload, indent=2))
print('Fit:', len(fit_ids), 'Internal validation:', len(val_ids), 'Development excluded:', len(excluded))
print('Split:', SPLITS)

In [ ]:
# Patch the public trainer with explicit deterministic seeding.
PATCHER = ROOT / 'scripts/prepare_v22_e016_clean_training_source.py'
CLEAN_TRAINER = Path('/kaggle/working/train_unet_transformer_clean.py')
subprocess.run([sys.executable, str(PATCHER), str(PUBLIC_TRAINER), str(CLEAN_TRAINER)], check=True)
print('Trainer:', CLEAN_TRAINER)

In [ ]:
# Final preflight. This cell must pass before training is launched.
assert len(list(TRAIN_DIR.glob('*.zarr'))) == 199
assert not (excluded & fit_ids | excluded & val_ids)
assert split['development_overlap'] is False
assert split['hidden_test_overlap'] is False
print('PREFLIGHT PASS: 138 fit / 34 internal validation / 27 development held out')

## Launch training

Run the next cell only after the preflight passes. The output checkpoint is selected using the clean internal validation set, never the 27 development samples.

In [ ]:
!python /kaggle/working/train_unet_transformer_clean.py --data-dir $TRAIN_DIR --splits /kaggle/working/dataset_splits_v22_e016_clean.json --split 0 --method unet_transformer_clean172_seed314159_v1 --epochs 500 --lr 1e-4 --batch-size 8 --num-workers 4 --unet-out-channels 32 --unet-layers 32,64,128 --downsample 1,4,4 --det-loss-weight 1.0 --det-neg-weight 0.01 --window-size 2 --pool-kernel-um 5.0 --single-gpu --seed 314159

In [ ]:
# Package a compact provenance record for download.
weights = Path('/kaggle/working/weights/unet_transformer_clean172_seed314159_v1/split_0/edge_predictor_best.pth')
assert weights.exists(), weights
record = dict(split)
record.update({'checkpoint_path': str(weights), 'checkpoint_sha256': hashlib.sha256(weights.read_bytes()).hexdigest(), 'graph_mutation': False, 'assignment': False})
Path('/kaggle/working/v22_e016_clean_checkpoint_record.json').write_text(json.dumps(record, indent=2))
print(record['checkpoint_sha256'])
print('Download v22_e016_clean_checkpoint_record.json and edge_predictor_best.pth')